# LoanGuard AI - MLOps Lab
## Data & Model Versioning + Registry

---

### Your Mission

You've just joined **LoanGuard AI**, a fintech startup that predicts loan defaults.
Their ML team has a serious problem: models keep breaking in production and nobody can explain why.
Predictions changed last week, accuracy dropped, and a roll back took 3 days.

**Your job:** Fix their MLOps stack - layer by layer - using data versioning, experiment tracking, model versioning, and a model registry.

---

### Lab Structure

| Task | Topic | Real-World Problem You Solve |
|------|-------|------------------------------|
| **Task 1** | Data Versioning with DVC | Reproduce a model that broke 2 months ago |
| **Task 2** | Experiment Tracking with MLflow | Compare 3 model candidates before deploying |
| **Task 3** | Model Versioning | Name and register a model so the team can roll back |
| **Task 4** | Registry Lifecycle | Promote a model through staging → production → archived |

---

### How the Lab Works

- Each task has a ** Beginner path** (fill in the blanks) and a ** Advanced path** (write from scratch)
- Use the ** Hint system** - try first, then reveal hints one at a time
- Run the ** Auto-check** cells to verify your work before moving on
- **You do not need to run DVC or MLflow CLI** - everything is done in Python

---

### Setup - Run this first

In [58]:
# installing required library for experiment tracking
!pip install mlflow

In [59]:
# --- SETUP - Run this cell before anything else ----------------------
import os, json, hashlib, shutil, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import pickle, time

warnings.filterwarnings("ignore")

# Lab workspace
LAB_DIR = Path("loanGuard_lab")
LAB_DIR.mkdir(exist_ok=True)
(LAB_DIR / "data").mkdir(exist_ok=True)
(LAB_DIR / "models").mkdir(exist_ok=True)
(LAB_DIR / "dvc_cache").mkdir(exist_ok=True)
MLFLOW_URI = f"sqlite:///{LAB_DIR}/mlflow.db"

print(" Setup complete - LoanGuard AI Lab is ready")
print(f"   Workspace : {LAB_DIR.resolve()}")
print(f"   MLflow DB : {MLFLOW_URI}")

 Setup complete - LoanGuard AI Lab is ready
   Workspace : /home/ciuser/jupyterteam3/students/Prasanna_Jupyter/DataModel/loanGuard_lab
   MLflow DB : sqlite:///loanGuard_lab/mlflow.db


### Simulated Data Generator

LoanGuard AI receives loan applications with customer financial data.
Run the cell below to simulate **three versions** of their dataset - representing how data changed over time.

>  **Don't modify this cell.** It simulates real-world upstream data changes you'll need to detect and version.

In [60]:
# ----- DATA SIMULATOR - Do not modify ----------------------------------
np.random.seed(42)

def make_loan_dataset(n=2000, schema_version="v1", seed=42):
    """Simulate LoanGuard's loan application data at different points in time."""
    rng = np.random.RandomState(seed)
    age            = rng.randint(22, 65, n)
    income         = rng.normal(55000, 20000, n).clip(15000, 200000)
    loan_amount    = rng.normal(12000, 5000, n).clip(1000, 50000)
    credit_score   = rng.normal(680, 80, n).clip(300, 850)
    employment_yrs = rng.randint(0, 30, n)
    debt_ratio     = rng.uniform(0.05, 0.65, n)

    # Default probability driven by risk factors
    risk = (
        -0.3 * (credit_score - 680) / 80
        + 0.4 * debt_ratio
        + 0.2 * (loan_amount / income)
        - 0.1 * employment_yrs / 10
    )
    default = (1 / (1 + np.exp(-risk * 3 + rng.normal(0, 0.5, n))) > 0.5).astype(int)

    df = pd.DataFrame({
        "age": age, "annual_income": income.round(2),
        "loan_amount": loan_amount.round(2), "credit_score": credit_score.round(1),
        "employment_years": employment_yrs, "debt_to_income": debt_ratio.round(4),
        "default": default
    })

    # v2: upstream adds a new column (common real-world schema drift)
    if schema_version in ("v2", "v3"):
        df["num_accounts"] = rng.randint(1, 10, n)

    # v3: a data quality issue introduces nulls (another common real-world problem)
    if schema_version == "v3":
        null_idx = rng.choice(n, size=int(n * 0.15), replace=False)
        df.loc[null_idx, "credit_score"] = np.nan

    return df

# Generate and save three dataset versions
datasets = {}
for ver, schema, seed in [("v1", "v1", 42), ("v2", "v2", 42), ("v3", "v3", 99)]:
    df = make_loan_dataset(schema_version=schema, seed=seed)
    path = LAB_DIR / "data" / f"loans_{ver}.csv"
    df.to_csv(path, index=False)
    datasets[ver] = df
    print(f"  loans_{ver}.csv  |  {len(df):,} rows  |  {df.shape[1]} cols  |  "
          f"{df['default'].mean():.1%} default rate  |  nulls: {df.isnull().sum().sum()}")

print("\n Three dataset versions ready")

  loans_v1.csv  |  2,000 rows  |  7 cols  |  58.1% default rate  |  nulls: 0
  loans_v2.csv  |  2,000 rows  |  8 cols  |  58.1% default rate  |  nulls: 0
  loans_v3.csv  |  2,000 rows  |  8 cols  |  54.6% default rate  |  nulls: 300

 Three dataset versions ready


---
# TASK 1 - Data Versioning with DVC
> Can you prove which data trained the broken model?
---

##  Task 1 - Data Versioning with DVC

### Scenario
> LoanGuard's model accuracy dropped from 87% to 71% last month. The engineering team says 'the code didn't change'. You suspect a silent data change. You need to (a) version the dataset snapshots so they can be reproduced later, (b) detect what changed between versions, and (c) implement schema validation to catch future changes automatically.

### Objective
By the end of this task you will be able to:
- Simulate DVC-style versioning using checksums and metadata (`.dvc` pointer files)
- Compare two dataset versions and identify schema/distribution drift
- Write a schema validator that raises an alert before a bad dataset enters the pipeline

---

###  Background - How DVC Versioning Works

DVC doesn't store data in Git. It stores a **tiny pointer file** (`.dvc`) containing:
- A **checksum** (MD5 hash) of the data file
- The file path and size
- The remote storage location

When you run `dvc pull`, DVC uses the checksum to fetch the exact data file.

In this task you'll **simulate this mechanism in pure Python** - the same logic DVC uses internally.

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  1A - Create a DVC-style pointer file

Create a function that takes a CSV file path and produces a `.dvc` metadata dictionary
containing: `md5`, `size`, `path`, `rows`, `cols`, and `created_at`.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `hashlib.md5()` to compute the file checksum. Open the file in binary mode (`'rb'`) and feed it to `.update()`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

For metadata: `os.path.getsize(path)` gives size in bytes. Load with `pd.read_csv()` and use `.shape` for rows/cols.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def create_dvc_pointer(csv_path):
    path = Path(csv_path)
    with open(path, 'rb') as f:
        md5 = hashlib.md5(f.read()).hexdigest()
    df = pd.read_csv(path)
    return {
        'md5': md5, 'path': str(path),
        'size': os.path.getsize(path),
        'rows': len(df), 'cols': df.shape[1],
        'created_at': datetime.now().isoformat()
    }
```

</details>

In [61]:
#  BEGINNER 1A - Create DVC-style pointer files
# ----------------------------------------------------------------------------------------

def create_dvc_pointer(csv_path):
    """
    Create a DVC-style metadata pointer for a dataset file.

    Parameters
    ----------
    csv_path : str or Path
        Path to the CSV file to version.

    Returns
    -------
    dict  with keys: md5, path, size, rows, cols, created_at
    """
    path = Path(csv_path)

    # Step 1: Compute MD5 checksum of the file
    with open(path, 'rb') as f:
        md5 = hashlib.md5(f.read()).hexdigest()

    # Step 2: Load the CSV to get row/column counts
    df = pd.read_csv(path)

    # Step 3: Build and return the metadata dictionary
    return {
        "md5": md5,
        "path": str(path),
        "size": os.path.getsize(path),   # file size in bytes
        "rows": len(df),                 # number of rows
        "cols": df.shape[1],             # number of columns
        "created_at": datetime.now().isoformat()
    }

# ----- Test it -----
pointer_v1 = create_dvc_pointer(LAB_DIR / "data" / "loans_v1.csv")
print("DVC pointer for loans_v1:")
print(json.dumps(pointer_v1, indent=2))

DVC pointer for loans_v1:
{
  "md5": "32749950b434fb1f5e15bc9573a4ccd0",
  "path": "loanGuard_lab/data/loans_v1.csv",
  "size": 76059,
  "rows": 2000,
  "cols": 7,
  "created_at": "2026-03-13T13:44:30.906099"
}


###  1B - Version all three datasets and save pointer files

Call your function on all three loan datasets and save each pointer as a `.dvc` JSON file.

In [62]:
#  BEGINNER 1B - Version all three datasets
# --------------------------------------------------

dvc_store = {}   # will hold {version: pointer_dict}

for version in ["v1", "v2", "v3"]:
    csv_path = LAB_DIR / "data" / f"loans_{version}.csv"

    # Step 1: Create pointer using your function above
    pointer = create_dvc_pointer(csv_path)

    # Step 2: Save pointer to a .dvc JSON file
    dvc_file = LAB_DIR / "data" / f"loans_{version}.csv.dvc"

    with open(dvc_file, "w") as f:
        json.dump(pointer, f, indent=2)

    dvc_store[version] = pointer

    print(f"  loans_{version}.csv  →  md5: {pointer['md5'][:12]}...  "
          f"({pointer['rows']} rows, {pointer['cols']} cols)")

print("\n All versions tracked")

  loans_v1.csv  →  md5: 32749950b434...  (2000 rows, 7 cols)
  loans_v2.csv  →  md5: 3b43198058ae...  (2000 rows, 8 cols)
  loans_v3.csv  →  md5: b8ac0fe8ebdd...  (2000 rows, 8 cols)

 All versions tracked


###  1C - Compare two versions and detect drift

Write a function that compares two DVC pointers and the underlying CSVs to detect:
- Schema changes (new/removed columns)
- Row count changes
- Statistical drift in numeric columns (mean shift > 5%)
- Data quality issues (null counts)

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Load both CSVs with `pd.read_csv()`. Compare `df1.columns` and `df2.columns` using set operations to find added/removed columns.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

For numeric drift, iterate over shared columns: `abs(df2[col].mean() - df1[col].mean()) / (df1[col].mean() + 1e-9)`. Flag anything above 0.05.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def compare_versions(ptr_a, ptr_b):
    df_a, df_b = pd.read_csv(ptr_a['path']), pd.read_csv(ptr_b['path'])
    cols_a, cols_b = set(df_a.columns), set(df_b.columns)
    report = {
        'added_cols':   list(cols_b - cols_a),
        'removed_cols': list(cols_a - cols_b),
        'row_delta':    ptr_b['rows'] - ptr_a['rows'],
        'null_delta':   int(df_b.isnull().sum().sum() - df_a.isnull().sum().sum()),
        'drift': {}
    }
    for col in cols_a & cols_b:
        if df_a[col].dtype in [float, int]:
            shift = abs(df_b[col].mean() - df_a[col].mean()) / (abs(df_a[col].mean()) + 1e-9)
            if shift > 0.05:
                report['drift'][col] = round(shift, 4)
    return report
```

</details>

In [63]:
#  BEGINNER 1C - Detect drift between dataset versions
# ----------------------------------------------------------

def compare_dataset_versions(ptr_a, ptr_b):
    """
    Compare two versioned datasets and produce a drift report.

    Returns a dict with:
      - added_cols    : columns present in b but not a
      - removed_cols  : columns present in a but not b
      - row_delta     : change in row count
      - null_delta    : change in total null count
      - drift         : dict of {col: relative_mean_shift} for drifted columns
    """
    df_a = pd.read_csv(ptr_a["path"])
    df_b = pd.read_csv(ptr_b["path"])

    cols_a = set(df_a.columns)
    cols_b = set(df_b.columns)

    report = {
        "added_cols": list(cols_b - cols_a),
        "removed_cols": list(cols_a - cols_b),
        "row_delta": ptr_b["rows"] - ptr_a["rows"],
        "null_delta": int(df_b.isnull().sum().sum() - df_a.isnull().sum().sum()),
        "drift": {}
    }

    # Detect numeric drift in shared columns
    shared_numeric = [
        c for c in cols_a & cols_b
        if df_a[c].dtype in [np.float64, np.int64]
    ]

    for col in shared_numeric:
        mean_a = df_a[col].mean()
        mean_b = df_b[col].mean()

        shift = abs(mean_b - mean_a) / (abs(mean_a) + 1e-9)

        if shift > 0.05:
            report["drift"][col] = round(shift, 4)

    return report


# ----- Compare v1 → v2, then v2 → v3 -----
print("═" * 55)
print("Comparing loans_v1  →  loans_v2")
print("═" * 55)
r12 = compare_dataset_versions(dvc_store["v1"], dvc_store["v2"])
print(json.dumps(r12, indent=2))

print("\n" + "═" * 55)
print("Comparing loans_v2  →  loans_v3  (the bad version!)")
print("═" * 55)
r23 = compare_dataset_versions(dvc_store["v2"], dvc_store["v3"])
print(json.dumps(r23, indent=2))

═══════════════════════════════════════════════════════
Comparing loans_v1  →  loans_v2
═══════════════════════════════════════════════════════
{
  "added_cols": [
    "num_accounts"
  ],
  "removed_cols": [],
  "row_delta": 0,
  "null_delta": 0,
  "drift": {}
}

═══════════════════════════════════════════════════════
Comparing loans_v2  →  loans_v3  (the bad version!)
═══════════════════════════════════════════════════════
{
  "added_cols": [],
  "removed_cols": [],
  "row_delta": 0,
  "null_delta": 300,
  "drift": {
    "default": 0.0594
  }
}


###  1D - Schema Validator

Write a schema validator that checks incoming data against a **golden schema** (captured from a known-good dataset).
It must raise a `ValueError` with a descriptive message if validation fails.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Store the golden schema as a dict: `{'columns': list, 'dtypes': dict, 'null_thresholds': dict}`. Compare incoming data against it.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Check three things in order: (1) all expected columns present, (2) null rate per column below threshold, (3) value ranges reasonable.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def build_golden_schema(df, null_tolerance=0.05):
    return {
        'columns': list(df.columns),
        'null_rates': {c: df[c].isnull().mean() for c in df.columns},
        'null_tolerance': null_tolerance,
        'ranges': {c: (df[c].min(), df[c].max())
                   for c in df.select_dtypes(include=np.number).columns}
    }

def validate_schema(df, schema):
    errors = []
    missing = set(schema['columns']) - set(df.columns)
    if missing: errors.append(f'Missing columns: {missing}')
    for col in schema['columns']:
        if col in df.columns:
            null_rate = df[col].isnull().mean()
            if null_rate > schema['null_tolerance']:
                errors.append(f'{col}: null rate {null_rate:.1%} > tolerance')
    if errors:
        raise ValueError('Schema validation failed:\n' + '\n'.join(errors))
    return True
```

</details>

In [64]:
#  BEGINNER 1D - Schema Validator
# ----------------------------------------------------------

def build_golden_schema(df, null_tolerance=0.05):
    """
    Capture the schema of a known-good dataset.

    Returns a dict with: columns, null_rates, null_tolerance, ranges
    """
    # YOUR CODE HERE
    return {}


def validate_schema(df, schema):
    """
    Validate a new DataFrame against a golden schema.
    Raise ValueError if validation fails, listing ALL errors found.
    """
    errors = []

    # Check 1: All expected columns are present
    # YOUR CODE HERE

    # Check 2: Null rate per column is within tolerance
    # YOUR CODE HERE

    if errors:
        raise ValueError("Schema validation FAILED:\n" + "\n".join(f"   {e}" for e in errors))

    print(" Schema validation passed")
    return True


# ----- Test: v1 is the golden schema -----
golden = build_golden_schema(datasets["v1"])
print("Golden schema columns:", golden.get("columns", []))
print()

print("Validating v2 (has extra column)...")
try:
    validate_schema(datasets["v2"], golden)
except ValueError as e:
    print(f"Caught validation error:\n{e}")

print()
print("Validating v3 (has nulls + extra column)...")
try:
    validate_schema(datasets["v3"], golden)
except ValueError as e:
    print(f"  Caught validation error:\n{e}")

Golden schema columns: []

Validating v2 (has extra column)...
 Schema validation passed

Validating v3 (has nulls + extra column)...
 Schema validation passed


In [65]:
#  BEGINNER 1D - Schema Validator
# ----------------------------------------------------------

def build_golden_schema(df, null_tolerance=0.05):
    """
    Capture the schema of a known-good dataset.

    Returns a dict with: columns, null_rates, null_tolerance, ranges
    """

    schema = {
        "columns": list(df.columns),
        "null_rates": {c: df[c].isnull().mean() for c in df.columns},
        "null_tolerance": null_tolerance,
        "ranges": {
            c: (df[c].min(), df[c].max())
            for c in df.select_dtypes(include=np.number).columns
        }
    }

    return schema


def validate_schema(df, schema):
    """
    Validate a new DataFrame against a golden schema.
    Raise ValueError if validation fails, listing ALL errors found.
    """

    errors = []

    # Check 1: All expected columns are present
    missing = set(schema["columns"]) - set(df.columns)

    if missing:
        errors.append(f"Missing columns: {missing}")

    # Check 2: Null rate per column is within tolerance
    for col in schema["columns"]:
        if col in df.columns:
            null_rate = df[col].isnull().mean()

            if null_rate > schema["null_tolerance"]:
                errors.append(
                    f"{col}: null rate {null_rate:.2%} exceeds tolerance"
                )

    if errors:
        raise ValueError(
            "Schema validation FAILED:\n" +
            "\n".join(f"   {e}" for e in errors)
        )

    print(" Schema validation passed")
    return True


# ----- Test: v1 is the golden schema -----
golden = build_golden_schema(datasets["v1"])
print("Golden schema columns:", golden.get("columns", []))
print()

print("Validating v2 (has extra column)...")
try:
    validate_schema(datasets["v2"], golden)
except ValueError as e:
    print(f"Caught validation error:\n{e}")

print()
print("Validating v3 (has nulls + extra column)...")
try:
    validate_schema(datasets["v3"], golden)
except ValueError as e:
    print(f"  Caught validation error:\n{e}")

Golden schema columns: ['age', 'annual_income', 'loan_amount', 'credit_score', 'employment_years', 'debt_to_income', 'default']

Validating v2 (has extra column)...
 Schema validation passed

Validating v3 (has nulls + extra column)...
  Caught validation error:
Schema validation FAILED:
   credit_score: null rate 15.00% exceeds tolerance


In [66]:
#  AUTO-CHECK - Run this cell to verify your work
passed = []
failed = []

try:
    assert pointer_v1.get('md5') and len(pointer_v1['md5']) == 32, 'md5 must be a 32-char hex string'
    passed.append(' DVC pointer created successfully')
except Exception as e:
    failed.append(('DVC pointer created', str(e)))

try:
    assert all(k in pointer_v1 for k in ['md5','path','size','rows','cols']), 'Missing keys in pointer'
    passed.append(' DVC pointer contains all required fields')
except Exception as e:
    failed.append(('DVC pointer has required keys', str(e)))

try:
    assert len(dvc_store) == 3, 'dvc_store should have v1, v2, v3'
    passed.append(' All three dataset versions tracked')
except Exception as e:
    failed.append(('All 3 versions stored', str(e)))

try:
    assert r23.get('null_delta', 0) > 0, 'Should detect null increase in v3'
    passed.append(' Data drift correctly detected between v2 and v3')
except Exception as e:
    failed.append(('Drift detected v2→v3', str(e)))

try:
    assert True, 'Manually verify the validator raised an error for v3'
    passed.append(' Schema validator check completed')
except Exception as e:
    failed.append(('Schema validator catches v3', str(e)))

print('\n' + '='*50)
print("Assignment Check Summary")
print('='*50)

print(f'\n Passed: {len(passed)}/{len(passed)+len(failed)}')

for p in passed:
    print(f'  {p}')

if failed:
    print('\n Some checks need attention:')
    for f,e in failed:
        print(f'   {f}: {e}')
else:
    print('\n Great job! All checks passed successfully.')


Assignment Check Summary

 Passed: 5/5
   DVC pointer created successfully
   DVC pointer contains all required fields
   All three dataset versions tracked
   Data drift correctly detected between v2 and v3
   Schema validator check completed

 Great job! All checks passed successfully.


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 1

Implement a complete DVC-style versioning system with the following requirements:

**Requirements:**
1. `create_dvc_pointer(path)` - MD5 checksum, full metadata, saved to `.dvc` file
2. `compare_dataset_versions(ptr_a, ptr_b)` - schema diff, row diff, null diff, per-column statistical drift (mean, std, min, max shifts)
3. `build_golden_schema(df)` + `validate_schema(df, schema)` - catches missing columns, null rate violations, and out-of-range values for numeric columns
4. `reproduce_dataset(version)` - given a version string, loads the correct CSV using the `.dvc` pointer file (simulating `dvc pull`)
5. Demonstrate: show that v3 fails validation, v1 passes, and `reproduce_dataset('v1')` returns the exact same checksum as the v1 pointer

Write all functions in the cell below with no scaffolding.

In [67]:
#  ADVANCED - Task 1 Full Implementation
# ---------------------------------------
# YOUR CODE HERE
# ==========================================================
# This program simulates basic features of a data versioning
# workflow similar to tools like DVC.
#
# Features:
# 1. Create DVC pointer
# 2. Compare dataset versions
# 3. Build a reference schema
# 4. Validate new datasets against the schema
# 5. Reproduce datasets using metadata
# ==========================================================

import hashlib
import json
import os
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np


# ---------------------------------------------------
# 1. Create DVC pointer
# ---------------------------------------------------
def create_dvc_pointer(path):

    path = Path(path)

    # Compute MD5 checksum
    with open(path, "rb") as f:
        md5 = hashlib.md5(f.read()).hexdigest()

    df = pd.read_csv(path)

    pointer = {
        "md5": md5,
        "path": str(path),
        "size": os.path.getsize(path),
        "rows": df.shape[0],
        "cols": df.shape[1],
        "created_at": datetime.now().isoformat()
    }

    # Save pointer as .dvc file
    dvc_file = path.with_suffix(path.suffix + ".dvc")

    with open(dvc_file, "w") as f:
        json.dump(pointer, f, indent=2)

    return pointer


# ---------------------------------------------------
# 2. Compare dataset versions
# ---------------------------------------------------
def compare_dataset_versions(ptr_a, ptr_b):

    df_a = pd.read_csv(ptr_a["path"])
    df_b = pd.read_csv(ptr_b["path"])

    cols_a = set(df_a.columns)
    cols_b = set(df_b.columns)

    report = {
        "added_cols": list(cols_b - cols_a),
        "removed_cols": list(cols_a - cols_b),
        "row_delta": df_b.shape[0] - df_a.shape[0],
        "null_delta": int(df_b.isnull().sum().sum() - df_a.isnull().sum().sum()),
        "drift": {}
    }

    shared_cols = cols_a & cols_b

    for col in shared_cols:

        if df_a[col].dtype in [np.float64, np.int64]:

            stats_a = {
                "mean": df_a[col].mean(),
                "std": df_a[col].std(),
                "min": df_a[col].min(),
                "max": df_a[col].max()
            }

            stats_b = {
                "mean": df_b[col].mean(),
                "std": df_b[col].std(),
                "min": df_b[col].min(),
                "max": df_b[col].max()
            }

            drift_stats = {}

            for stat in stats_a:
                base = abs(stats_a[stat]) + 1e-9
                shift = abs(stats_b[stat] - stats_a[stat]) / base

                if shift > 0.05:
                    drift_stats[stat] = round(shift, 4)

            if drift_stats:
                report["drift"][col] = drift_stats

    return report


# ---------------------------------------------------
# 3. Build Golden Schema
# ---------------------------------------------------
def build_golden_schema(df, null_tolerance=0.05):

    schema = {
        "columns": list(df.columns),
        "null_tolerance": null_tolerance,
        "null_rates": {c: df[c].isnull().mean() for c in df.columns},
        "ranges": {
            c: (df[c].min(), df[c].max())
            for c in df.select_dtypes(include=np.number).columns
        }
    }

    return schema


# ---------------------------------------------------
# 4. Validate Schema
# ---------------------------------------------------
def validate_schema(df, schema):

    errors = []

    # Missing columns
    missing = set(schema["columns"]) - set(df.columns)

    if missing:
        errors.append(f"Missing columns: {missing}")

    # Null rate check
    for col in schema["columns"]:

        if col in df.columns:

            null_rate = df[col].isnull().mean()

            if null_rate > schema["null_tolerance"]:
                errors.append(
                    f"{col} null rate {null_rate:.2%} exceeds tolerance"
                )

    # Numeric range check
    for col, (min_val, max_val) in schema["ranges"].items():

        if col in df.columns:

            if df[col].min() < min_val or df[col].max() > max_val:
                errors.append(
                    f"{col} values out of expected range"
                )

    if errors:
        raise ValueError(
            "Schema validation FAILED:\n" +
            "\n".join(errors)
        )

    print("Schema validation passed")
    return True


# ---------------------------------------------------
# 5. Reproduce dataset (simulate dvc pull)
# ---------------------------------------------------
def reproduce_dataset(version):

    dvc_file = LAB_DIR / "data" / f"loans_{version}.csv.dvc"

    with open(dvc_file) as f:
        pointer = json.load(f)

    df = pd.read_csv(pointer["path"])

    # Verify checksum
    with open(pointer["path"], "rb") as f:
        md5 = hashlib.md5(f.read()).hexdigest()

    if md5 != pointer["md5"]:
        raise ValueError("Dataset checksum mismatch!")

    return df


# ---------------------------------------------------
# Demonstration
# ---------------------------------------------------

# Create pointers
ptr_v1 = create_dvc_pointer(LAB_DIR / "data" / "loans_v1.csv")
ptr_v2 = create_dvc_pointer(LAB_DIR / "data" / "loans_v2.csv")
ptr_v3 = create_dvc_pointer(LAB_DIR / "data" / "loans_v3.csv")

# Compare datasets
print("Dataset comparison v2 → v3")
report = compare_dataset_versions(ptr_v2, ptr_v3)
print(json.dumps(report, indent=2))


# Build schema from v1
df_v1 = pd.read_csv(ptr_v1["path"])
schema = build_golden_schema(df_v1)

print("\nValidating v1")
validate_schema(df_v1, schema)


print("\nValidating v3 (expected failure)")
try:
    df_v3 = pd.read_csv(ptr_v3["path"])
    validate_schema(df_v3, schema)
except ValueError as e:
    print(e)


# Reproduce dataset
print("\nReproducing dataset v1")
df_reproduced = reproduce_dataset("v1")

# Verify checksum again
with open(ptr_v1["path"], "rb") as f:
    md5 = hashlib.md5(f.read()).hexdigest()

print("Checksum match:", md5 == ptr_v1["md5"])

Dataset comparison v2 → v3
{
  "added_cols": [],
  "removed_cols": [],
  "row_delta": 0,
  "null_delta": 300,
  "drift": {
    "default": {
      "mean": 0.0594
    },
    "loan_amount": {
      "max": 0.124
    },
    "credit_score": {
      "min": 0.1951
    }
  }
}

Validating v1
Schema validation passed

Validating v3 (expected failure)
Schema validation FAILED:
credit_score null rate 15.00% exceeds tolerance
loan_amount values out of expected range
credit_score values out of expected range

Reproducing dataset v1
Checksum match: True


---
# TASK 2 - Experiment Tracking with MLflow
> Which model should go to production - and can you prove it?
---

##  Task 2 - Experiment Tracking with MLflow

### Scenario
> LoanGuard's data science lead trained three models last sprint - Logistic Regression, Random Forest, and Gradient Boosting. She can't remember which hyperparameters she used for the best one, and the comparison was done in a Slack message that got deleted. You need to run all three as tracked experiments so the decision is transparent and auditable.

### Objective
By the end of this task you will be able to:
- Create an MLflow experiment and log runs with parameters, metrics, and artifacts
- Compare multiple model runs programmatically
- Identify the best model using a business-relevant metric (F1 score for imbalanced default data)
- Log the training dataset version alongside model runs for full traceability

---

###  Background - MLflow Tracking

MLflow Tracking records every experiment run as a **first-class object** with:
- **Parameters** - hyperparameters, config values
- **Metrics** - accuracy, F1, AUC, etc.
- **Artifacts** - model files, plots, data samples
- **Tags** - free-form metadata (dataset version, author, etc.)

Each run gets a unique `run_id` that links it permanently to what produced it.

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  2A - Set up MLflow and prepare data

Configure MLflow tracking and split the versioned dataset into train/test sets.

In [68]:
#  BEGINNER 2A - Configure MLflow and prepare data
# ----------------------------------------------------

# Step 1: Set the MLflow tracking URI to your local SQLite database
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Step 2: Create (or get) an experiment called "loandefault_prediction"
experiment = mlflow.set_experiment("loandefault_prediction")
experiment_id = experiment.experiment_id

# Step 3: Load loans_v1.csv and split into train / test (80/20, random_state=42)
df = pd.read_csv(LAB_DIR / "data" / "loans_v1.csv").dropna()

FEATURES = ["age", "annual_income", "loan_amount",
            "credit_score", "employment_years", "debt_to_income"]
TARGET   = "default"

# Prepare feature matrix and target
X = df[FEATURES]
y = df[TARGET]

# Train / Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Experiment ready")
print(f"Train size : {len(X_train):,}")
print(f"Test size  : {len(X_test):,}")
print(f"Default rate (train): {y_train.mean():.1%}")

Experiment ready
Train size : 1,600
Test size  : 400
Default rate (train): 58.1%


###  2B - Log a single MLflow run

Write a function that trains one model and logs everything to MLflow: params, metrics, the model artifact, and a tag for the dataset version.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `with mlflow.start_run(run_name=...) as run:` to start a run. Inside: `mlflow.log_param(key, val)` and `mlflow.log_metric(key, val)`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Log the model with `mlflow.sklearn.log_model(model, artifact_path='model')`. Add tags with `mlflow.set_tag(key, val)`.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def run_experiment(model, model_name, params, X_tr, X_te, y_tr, y_te, dataset_version):
    with mlflow.start_run(run_name=model_name) as run:
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        proba = model.predict_proba(X_te)[:, 1]

        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy':  accuracy_score(y_te, preds),
            'precision': precision_score(y_te, preds),
            'recall':    recall_score(y_te, preds),
            'f1':        f1_score(y_te, preds),
            'auc':       roc_auc_score(y_te, proba)
        })
        mlflow.set_tag('dataset_version', dataset_version)
        mlflow.set_tag('model_type', model_name)
        mlflow.sklearn.log_model(model, 'model')
        return run.info.run_id, f1_score(y_te, preds)
```

</details>

In [69]:
import warnings
warnings.filterwarnings("ignore")

In [70]:
#  BEGINNER 2B - Log a single MLflow run
# -----------------------------------------------------

def run_experiment(model, model_name, params, X_tr, X_te, y_tr, y_te, dataset_version):
    """
    Train a model and log a complete MLflow run.

    Must log:
      - All items in `params` dict as MLflow parameters
      - accuracy, precision, recall, f1, auc as MLflow metrics
      - The fitted model as an artifact
      - 'dataset_version' and 'model_type' as tags

    Returns
    -------
    (run_id : str, f1 : float)
    """
    with mlflow.start_run(run_name=model_name) as run:

        # Step 1: Train the model
        model.fit(X_tr, y_tr)

        # Step 2: Generate predictions and probabilities
        preds = model.predict(X_te)
        proba = model.predict_proba(X_te)[:, 1]

        # Step 3: Log parameters
        mlflow.log_params(params)

        # Step 4: Log metrics - accuracy, precision, recall, f1, auc
        mlflow.log_metric("accuracy",  accuracy_score(y_te, preds))
        mlflow.log_metric("precision", precision_score(y_te, preds))
        mlflow.log_metric("recall",    recall_score(y_te, preds))
        mlflow.log_metric("f1",        f1_score(y_te, preds))
        mlflow.log_metric("auc",       roc_auc_score(y_te, proba))

        # Step 5: Set tags for dataset_version and model_type
        mlflow.set_tag("dataset_version", dataset_version)
        mlflow.set_tag("model_type", model_name)

        # Step 6: Log the model artifact
        mlflow.sklearn.log_model(model, name="model")#artifact path

        f1 = f1_score(y_te, preds)
        return run.info.run_id, f1


# ----- Test with one model -----
test_model  = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
test_params = {"C": 1.0, "max_iter": 1000, "solver": "lbfgs"}

run_id, f1  = run_experiment(
    test_model, "LogisticRegression_test", test_params,
    X_train, X_test, y_train, y_test, dataset_version="v1"
)

print(f"Run ID : {run_id}")
print(f"F1     : {f1:.4f}")

2026/03/13 13:45:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID : 57571980c52b4811abcd58e774bc2e13
F1     : 0.8821


###  2C - Run all three candidate models

Run all three model types with the parameters below and collect their run IDs and F1 scores.

In [71]:
#  BEGINNER 2C - Run all three candidate models
# ------------------------------------------------------

# Three candidate models with their hyperparameter configs
candidates = [
    (
        LogisticRegression(C=0.5, max_iter=1000, random_state=42),
        "LogisticRegression",
        {"model": "LogisticRegression", "C": 0.5, "max_iter": 1000}
    ),
    (
        RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
        "RandomForest",
        {"model": "RandomForest", "n_estimators": 100, "max_depth": 6}
    ),
    (
        GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                   max_depth=4, random_state=42),
        "GradientBoosting",
        {"model": "GradientBoosting", "n_estimators": 100,
         "learning_rate": 0.1, "max_depth": 4}
    ),
]

# Loop through candidate models
results = []

for model, name, params in candidates:

    run_id, f1 = run_experiment(
        model,
        name,
        params,
        X_train,
        X_test,
        y_train,
        y_test,
        dataset_version="v1"
    )

    results.append({
        "name": name,
        "run_id": run_id,
        "f1": f1
    })


# ----- Print comparison -----
print("\n" + "═"*55)
print(f"{'Model':<25} {'F1 Score':>10} {'Run ID':>15}")
print("═"*55)

for r in results:
    print(f"{r['name']:<25} {r['f1']:>10.4f}  {r['run_id'][:12]}...")

print("═"*55)

if results:
    best = max(results, key=lambda x: x["f1"])
    print(f"\nBest model: {best['name']}  (F1 = {best['f1']:.4f})")

    BEST_RUN_ID = best["run_id"]
    BEST_MODEL_NAME = best["name"]

2026/03/13 13:45:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 13:45:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 13:45:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec


═══════════════════════════════════════════════════════
Model                       F1 Score          Run ID
═══════════════════════════════════════════════════════
LogisticRegression            0.8783  fcdf035dcfab...
RandomForest                  0.8690  da70a2653da5...
GradientBoosting              0.8684  88f78cb1784e...
═══════════════════════════════════════════════════════

Best model: LogisticRegression  (F1 = 0.8783)


###  2D - Retrieve the best model from MLflow

Use the MLflow client to load the best model back from the tracking server - as if you're another engineer picking up the work.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `MlflowClient()` to interact with the tracking server. `client.get_run(run_id)` retrieves run metadata.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

`mlflow.sklearn.load_model(f'runs:/{run_id}/model')` loads the model artifact from a run.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
client = MlflowClient()
run    = client.get_run(BEST_RUN_ID)
print(run.data.metrics)
model  = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")
preds  = model.predict(X_test)
```

</details>

In [41]:
#  BEGINNER 2D - Retrieve and verify the best model
# ------------------------------------------------------

client = MlflowClient()

# Step 1: Retrieve the run metadata for BEST_RUN_ID
best_run = client.get_run(BEST_RUN_ID)

# Step 2: Print what was logged
if best_run:
    print("Run metadata:")
    print(f"  Model type      : {best_run.data.tags.get('model_type')}")
    print(f"  Dataset version : {best_run.data.tags.get('dataset_version')}")
    print(f"  F1 score        : {best_run.data.metrics.get('f1', 0):.4f}")
    print(f"  Params          : {best_run.data.params}")

# Step 3: Load the model artifact
loaded_model = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")

# Step 4: Verify it produces the same predictions
if loaded_model is not None:
    reloaded_preds = loaded_model.predict(X_test)
    print(f"\nVerification F1: {f1_score(y_test, reloaded_preds):.4f}")
    print(" Model successfully retrieved from MLflow")

Run metadata:
  Model type      : LogisticRegression
  Dataset version : v1
  F1 score        : 0.8783
  Params          : {'model': 'LogisticRegression', 'C': '0.5', 'max_iter': '1000'}



Verification F1: 0.8783
 Model successfully retrieved from MLflow


In [72]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert mlflow.get_experiment_by_name("loandefault_prediction") is not None, "Create experiment 'loandefault_prediction'"
    passed.append('MLflow experiment exists')
except Exception as e:
    failed.append(('MLflow experiment exists', str(e)))

try:
    assert len(results) == 3, 'results list should have 3 entries'
    passed.append('3 candidates ran')
except Exception as e:
    failed.append(('3 candidates ran', str(e)))

try:
    assert 'BEST_RUN_ID' in dir() or 'BEST_RUN_ID' in globals(), 'BEST_RUN_ID must be defined'
    passed.append('Best run ID set')
except Exception as e:
    failed.append(('Best run ID set', str(e)))

try:
    assert loaded_model is not None, 'loaded_model should not be None'
    passed.append('Model reloads OK')
except Exception as e:
    failed.append(('Model reloads OK', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed:
    print(f'  {p}')

if failed:
    print('\nFailed:')
    for f,e in failed:
        print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 4/4
  MLflow experiment exists
  3 candidates ran
  Best run ID set
  Model reloads OK

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 2

Implement a full experiment comparison pipeline:

**Requirements:**
1. Configure MLflow with the local SQLite URI
2. Write `run_experiment()` that logs: params, 5 metrics (accuracy/precision/recall/f1/auc), model artifact, dataset version tag, and training timestamp tag
3. Run all three candidate models (configurations given in Beginner 2C)
4. Write `find_best_run(experiment_name, metric='f1')` that queries MLflow programmatically (using `MlflowClient.search_runs()`) and returns the run with the highest value of `metric`
5. Load the best model, re-evaluate on test set, and confirm metrics match what was logged
6. **Bonus:** Add a second experiment round where you retrain the best model type on `loans_v2.csv` and log the dataset version change

In [73]:
#  ADVANCED - Task 2 Full Implementation
# ----------------------------------------------------------

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from datetime import datetime
import pandas as pd


# ----------------------------------------------------------
# 1. Configure MLflow
# ----------------------------------------------------------

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("loandefault_prediction")


# ----------------------------------------------------------
# 2. Load dataset
# ----------------------------------------------------------

df = pd.read_csv(LAB_DIR / "data" / "loans_v1.csv").dropna()

FEATURES = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "employment_years",
    "debt_to_income",
]

TARGET = "default"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# ----------------------------------------------------------
# 3. Experiment runner
# ----------------------------------------------------------

def run_experiment(model, name, params, dataset_version):

    with mlflow.start_run(run_name=name) as run:

        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        # log parameters
        mlflow.log_params(params)

        # log metrics
        metrics = {
            "accuracy": accuracy_score(y_test, preds),
            "precision": precision_score(y_test, preds),
            "recall": recall_score(y_test, preds),
            "f1": f1_score(y_test, preds),
            "auc": roc_auc_score(y_test, proba),
        }

        mlflow.log_metrics(metrics)

        # log tags
        mlflow.set_tag("dataset_version", dataset_version)
        mlflow.set_tag("training_time", datetime.now().isoformat())
        mlflow.set_tag("model_type", name)

        # log model
        mlflow.sklearn.log_model(model, name="model")

        return run.info.run_id, metrics["f1"]


# ----------------------------------------------------------
# 4. Candidate models
# ----------------------------------------------------------

candidates = [
    (
        LogisticRegression(C=0.5, max_iter=1000, random_state=42),
        "LogisticRegression",
        {"model": "LogisticRegression", "C": 0.5, "max_iter": 1000},
    ),
    (
        RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
        "RandomForest",
        {"model": "RandomForest", "n_estimators": 100, "max_depth": 6},
    ),
    (
        GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42
        ),
        "GradientBoosting",
        {
            "model": "GradientBoosting",
            "n_estimators": 100,
            "learning_rate": 0.1,
            "max_depth": 4,
        },
    ),
]


results = []

for model, name, params in candidates:

    run_id, f1 = run_experiment(model, name, params, dataset_version="v1")

    results.append({"name": name, "run_id": run_id, "f1": f1})


# ----------------------------------------------------------
# 5. Find best run using MLflowClient
# ----------------------------------------------------------

def find_best_run(experiment_name, metric="f1"):

    client = MlflowClient()

    exp = client.get_experiment_by_name(experiment_name)

    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=[f"metrics.{metric} DESC"],
    )

    return runs[0]


best_run = find_best_run("loandefault_prediction")

print("Best run ID:", best_run.info.run_id)
print("Best F1:", best_run.data.metrics["f1"])


# ----------------------------------------------------------
# 6. Load best model
# ----------------------------------------------------------

best_model = mlflow.sklearn.load_model(f"runs:/{best_run.info.run_id}/model")


# ----------------------------------------------------------
# 7. Re-evaluate model
# ----------------------------------------------------------

preds = best_model.predict(X_test)
proba = best_model.predict_proba(X_test)[:, 1]

print("\nVerification metrics")

print("Accuracy :", accuracy_score(y_test, preds))
print("Precision:", precision_score(y_test, preds))
print("Recall   :", recall_score(y_test, preds))
print("F1       :", f1_score(y_test, preds))
print("AUC      :", roc_auc_score(y_test, proba))


# ----------------------------------------------------------
# 8. BONUS – Retrain best model on v2 dataset
# ----------------------------------------------------------

df_v2 = pd.read_csv(LAB_DIR / "data" / "loans_v2.csv").dropna()

X2 = df_v2[FEATURES]
y2 = df_v2[TARGET]

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

best_model_type = best_run.data.tags["model_type"]

if best_model_type == "LogisticRegression":
    model = LogisticRegression(C=0.5, max_iter=1000, random_state=42)

elif best_model_type == "RandomForest":
    model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)

else:
    model = GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42
    )


with mlflow.start_run(run_name=f"{best_model_type}_v2"):

    model.fit(X_train2, y_train2)

    preds = model.predict(X_test2)
    proba = model.predict_proba(X_test2)[:, 1]

    mlflow.log_metric("f1", f1_score(y_test2, preds))

    mlflow.set_tag("dataset_version", "v2")

    mlflow.sklearn.log_model(model, name="model")

print("\nSecond experiment round completed on dataset v2")

2026/03/13 13:45:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 13:45:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 13:45:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

Best run ID: 57571980c52b4811abcd58e774bc2e13
Best F1: 0.8820960698689956



Verification metrics
Accuracy : 0.865
Precision: 0.8938053097345132
Recall   : 0.8706896551724138
F1       : 0.8820960698689956
AUC      : 0.937217775041051


2026/03/13 13:45:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Second experiment round completed on dataset v2


---
# TASK 3 - Model Versioning
> Give your model a name that tells the whole story
---

##  Task 3 - Model Versioning - Naming, Metadata & Semantic Versions

### Scenario
> LoanGuard's best model from Task 2 needs to be packaged for the team. The previous engineer just saved files as `model.pkl` with no metadata. When it broke, nobody knew what version was running or how to go back. You need to create a versioned model artifact with full metadata and implement semantic versioning logic.

### Objective
By the end of this task you will be able to:
- Save a model artifact with complete metadata (dataset version, metrics, git hash, author)
- Implement semantic versioning logic (MAJOR.MINOR.PATCH) for model releases
- Register a named, versioned model in MLflow Model Registry
- Retrieve a specific model version by name

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  3A - Save a model with complete metadata

Save the best model from Task 2 alongside a metadata sidecar file that fully describes it.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Save the model with `pickle.dump()`. Create a separate `_metadata.json` file in the same folder with all relevant context.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Metadata should include: model_name, version, dataset_version, metrics, params, trained_at, and a simulated git_hash (`hashlib.md5(str(time.time()).encode()).hexdigest()[:8]`).

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def save_versioned_model(model, name, version, metrics, params, dataset_version):
    path = LAB_DIR / 'models' / f'{name}-{version}'
    path.mkdir(exist_ok=True)
    with open(path / 'model.pkl', 'wb') as f:
        pickle.dump(model, f)
    meta = {
        'name': name, 'version': version,
        'dataset_version': dataset_version,
        'metrics': metrics, 'params': params,
        'git_hash': hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
        'trained_at': datetime.now().isoformat(),
        'mlflow_run_id': BEST_RUN_ID
    }
    with open(path / 'metadata.json', 'w') as f:
        json.dump(meta, f, indent=2)
    return path, meta
```

</details>

In [74]:
#  BEGINNER 3A - Save versioned model with metadata
# ------------------------------------------------------------

import pickle
import json
import hashlib
import time
from datetime import datetime
from pathlib import Path
from mlflow.tracking import MlflowClient

def save_versioned_model(model, name, version, metrics, params,
                         dataset_version, run_id):
    """
    Save a model and its complete metadata sidecar.

    Creates:
      models/{name}-{version}/
        model.pkl        ← serialised model
        metadata.json    ← full provenance record

    Returns
    -------
    (path : Path, metadata : dict)
    """

    # Step 1: Create output directory
    path = LAB_DIR / "models" / f"{name}-{version}"
    path.mkdir(parents=True, exist_ok=True)

    # Step 2: Save model with pickle
    with open(path / "model.pkl", "wb") as f:
        pickle.dump(model, f)

    # Step 3: Build metadata dict
    metadata = {
        "name": name,
        "version": version,
        "dataset_version": dataset_version,
        "metrics": metrics,
        "params": params,
        "git_hash": hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
        "trained_at": datetime.now().isoformat(),
        "mlflow_run_id": run_id,
        "author": "loanGuard-ml-team"
    }

    # Step 4: Save metadata as JSON
    with open(path / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    return path, metadata


# ----- Save the best model from Task 2 -----

client = MlflowClient()

best_run_data = client.get_run(BEST_RUN_ID)

best_metrics = best_run_data.data.metrics
best_params  = best_run_data.data.params

best_fitted = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")

model_path, model_meta = save_versioned_model(
    model           = best_fitted,
    name            = "loan-default-detector",
    version         = "v1.0.0",
    metrics         = best_metrics,
    params          = best_params,
    dataset_version = "v1",
    run_id          = BEST_RUN_ID
)

print(f"Saved to: {model_path}")

print("\nMetadata:")
print(json.dumps(model_meta, indent=2))

Saved to: loanGuard_lab/models/loan-default-detector-v1.0.0

Metadata:
{
  "name": "loan-default-detector",
  "version": "v1.0.0",
  "dataset_version": "v1",
  "metrics": {
    "accuracy": 0.86,
    "precision": 0.8859649122807017,
    "recall": 0.8706896551724138,
    "f1": 0.8782608695652174,
    "auc": 0.936858579638752
  },
  "params": {
    "model": "LogisticRegression",
    "C": "0.5",
    "max_iter": "1000"
  },
  "git_hash": "4ed8e69d",
  "trained_at": "2026-03-13T13:45:54.734921",
  "mlflow_run_id": "fcdf035dcfab4f2da92566324fb6e02d",
  "author": "loanGuard-ml-team"
}


###  3B - Semantic versioning logic

Implement a function that determines the correct next version number given the type of change.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Parse the version string by splitting on '.': `major, minor, patch = version.lstrip('v').split('.')`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

For 'major' bump: increment major, reset minor and patch to 0. For 'minor': increment minor, reset patch. For 'patch': just increment patch.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def bump_version(current, bump_type):
    major, minor, patch = [int(x) for x in current.lstrip('v').split('.')]
    if bump_type == 'major':   return f'v{major+1}.0.0'
    elif bump_type == 'minor': return f'v{major}.{minor+1}.0'
    elif bump_type == 'patch': return f'v{major}.{minor}.{patch+1}'
```

</details>

In [75]:
#  BEGINNER 3B - Semantic versioning
# ------------------------------------------------------------

def bump_version(current_version, bump_type):
    """
    Increment a semantic version string.

    Parameters
    ----------
    current_version : str  e.g. 'v1.2.3'
    bump_type       : str  one of 'major', 'minor', 'patch'

    Returns
    -------
    str  e.g. 'v2.0.0' for a major bump on 'v1.2.3'

    Rules
    -----
    major → increment MAJOR, reset MINOR and PATCH to 0
    minor → increment MINOR, reset PATCH to 0
    patch → increment PATCH only
    """

    # Step 1: Parse current_version into major, minor, patch integers
    major, minor, patch = [int(x) for x in current_version.lstrip('v').split('.')]

    # Step 2: Apply bump logic
    if bump_type == "major":
        major += 1
        minor = 0
        patch = 0

    elif bump_type == "minor":
        minor += 1
        patch = 0

    elif bump_type == "patch":
        patch += 1

    return f"v{major}.{minor}.{patch}"


# ----- Test all bump types -----
start = "v1.0.0"
print(f"Start version      : {start}")
print(f"After patch bump   : {bump_version(start, 'patch')}   ← threshold tweak")
print(f"After minor bump   : {bump_version(start, 'minor')}   ← new training data")
print(f"After major bump   : {bump_version(start, 'major')}   ← new architecture")
print()

# Simulate LoanGuard's version history
history = [
    ("v1.0.0", "Initial release - Random Forest on loans_v1"),
    (bump_version("v1.0.0", "patch"), "Fixed decision threshold from 0.5 → 0.46"),
    (bump_version("v1.0.1", "minor"), "Retrained on loans_v2 with new num_accounts feature"),
    (bump_version("v1.1.0", "major"), "Switched to GradientBoosting - breaking output format change"),
]

print("LoanGuard model version history:")

for ver, desc in history:
    print(f"  {ver:<12} {desc}")

Start version      : v1.0.0
After patch bump   : v1.0.1   ← threshold tweak
After minor bump   : v1.1.0   ← new training data
After major bump   : v2.0.0   ← new architecture

LoanGuard model version history:
  v1.0.0       Initial release - Random Forest on loans_v1
  v1.0.1       Fixed decision threshold from 0.5 → 0.46
  v1.1.0       Retrained on loans_v2 with new num_accounts feature
  v2.0.0       Switched to GradientBoosting - breaking output format change


###  3C - Register in MLflow Model Registry

Register your best model in the MLflow Model Registry under a named model, then retrieve it by name.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `mlflow.register_model(model_uri, name)` where `model_uri = f'runs:/{run_id}/model'`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

After registration, use `MlflowClient().get_registered_model(name)` to inspect it. `client.get_latest_versions(name)` lists all versions.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
model_uri = f"runs:/{BEST_RUN_ID}/model"
reg = mlflow.register_model(model_uri, "LoanDefaultDetector")
print(reg.version, reg.name)

client  = MlflowClient()
version = client.get_latest_versions("LoanDefaultDetector")[0]
loaded  = mlflow.sklearn.load_model(f"models:/LoanDefaultDetector/{version.version}")
```

</details>

In [76]:
#  BEGINNER 3C - Register model in MLflow Registry
# ----------------------------------------------------------

REGISTRY_NAME = "LoanDefaultDetector"

# Step 1: Register the best model run in the MLflow registry
model_uri = f"runs:/{BEST_RUN_ID}/model"

registered = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTRY_NAME
)

if registered:
    print(f"Registered model : {registered.name}")
    print(f"Version          : {registered.version}")
    print(f"Source run       : {registered.source}")


# Step 2: Retrieve and inspect all registered versions
client = MlflowClient()

versions = client.get_latest_versions(REGISTRY_NAME)

print("\nRegistered versions:")
for v in versions:
    print(f"  Version {v.version}  | Stage: {v.current_stage}")


# Step 3: Load the registered model by name and verify
latest_version = versions[0]

registry_model = mlflow.sklearn.load_model(
    f"models:/{REGISTRY_NAME}/{latest_version.version}"
)

if registry_model is not None:
    verify_preds = registry_model.predict(X_test)

    print(f"\nVerification F1 from registry: {f1_score(y_test, verify_preds):.4f}")
    print(" Model successfully retrieved from registry by name")

Registered model 'LoanDefaultDetector' already exists. Creating a new version of this model...
2026/03/13 13:46:15 WARNING mlflow.tracking._model_registry.fluent: Run with id fcdf035dcfab4f2da92566324fb6e02d has no artifacts at artifact path 'model', registering model based on models:/m-8d5905685169456e80edecf3c06f10ab instead


Registered model : LoanDefaultDetector
Version          : 6
Source run       : models:/m-8d5905685169456e80edecf3c06f10ab

Registered versions:
  Version 6  | Stage: None
  Version 5  | Stage: Archived
  Version 4  | Stage: Production

Verification F1 from registry: 0.8783
 Model successfully retrieved from registry by name


Created version '6' of model 'LoanDefaultDetector'.


In [77]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert (LAB_DIR / 'models' / 'loan-default-detector-v1.0.0' / 'model.pkl').exists(), 'model.pkl not found'
    passed.append('Versioned model saved')
except Exception as e:
    failed.append(('Versioned model saved', str(e)))

try:
    assert (LAB_DIR / 'models' / 'loan-default-detector-v1.0.0' / 'metadata.json').exists(), 'metadata.json not found'
    passed.append('Metadata saved')
except Exception as e:
    failed.append(('Metadata saved', str(e)))

try:
    assert bump_version('v1.0.0', 'patch') == 'v1.0.1', 'patch bump failed'
    passed.append('Patch bump correct')
except Exception as e:
    failed.append(('Patch bump correct', str(e)))

try:
    assert bump_version('v1.0.0', 'minor') == 'v1.1.0', 'minor bump failed'
    passed.append('Minor bump correct')
except Exception as e:
    failed.append(('Minor bump correct', str(e)))

try:
    assert bump_version('v1.0.0', 'major') == 'v2.0.0', 'major bump failed'
    passed.append('Major bump correct')
except Exception as e:
    failed.append(('Major bump correct', str(e)))

try:
    assert registered is not None, 'registered model should not be None'
    passed.append('Model registered')
except Exception as e:
    failed.append(('Model registered', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 6/6
  Versioned model saved
  Metadata saved
  Patch bump correct
  Minor bump correct
  Major bump correct
  Model registered

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 3

Build a complete model versioning and registry system:

**Requirements:**
1. `save_versioned_model()` - saves model + full metadata sidecar (all fields from Beginner 3A)
2. `bump_version()` - semantic version logic with validation (reject unknown bump types)
3. `ModelVersionHistory` class with methods:
   - `record(version, change_type, description, metrics)` - adds an entry
   - `get_changelog()` - returns formatted changelog string
   - `get_version_for_rollback(n_versions_back)` - returns the version string from n releases ago
4. Register model in MLflow Registry, add a description, and add the semantic version as a tag
5. **Demonstrate rollback:** Simulate a production failure, use `get_version_for_rollback(1)` to identify the previous version, load it from the local store, and confirm it produces different (better) metrics than the failed version

In [78]:
#  ADVANCED - Task 3 Full Implementation
# ----------------------------------------------------------

import pickle
import json
import hashlib
import time
from datetime import datetime
from pathlib import Path
from mlflow.tracking import MlflowClient
import mlflow
import mlflow.sklearn


# ----------------------------------------------------------
# 1. Save versioned model with metadata
# ----------------------------------------------------------

def save_versioned_model(model, name, version, metrics, params, dataset_version, run_id):

    path = LAB_DIR / "models" / f"{name}-{version}"
    path.mkdir(parents=True, exist_ok=True)

    # save model
    with open(path / "model.pkl", "wb") as f:
        pickle.dump(model, f)

    metadata = {
        "name": name,
        "version": version,
        "dataset_version": dataset_version,
        "metrics": metrics,
        "params": params,
        "git_hash": hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
        "trained_at": datetime.now().isoformat(),
        "mlflow_run_id": run_id,
        "author": "loanGuard-ml-team"
    }

    with open(path / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    return path, metadata


# ----------------------------------------------------------
# 2. Semantic version bump logic with validation
# ----------------------------------------------------------

def bump_version(current_version, bump_type):

    if bump_type not in ["major", "minor", "patch"]:
        raise ValueError("bump_type must be 'major', 'minor', or 'patch'")

    major, minor, patch = [int(x) for x in current_version.lstrip("v").split(".")]

    if bump_type == "major":
        major += 1
        minor = 0
        patch = 0

    elif bump_type == "minor":
        minor += 1
        patch = 0

    elif bump_type == "patch":
        patch += 1

    return f"v{major}.{minor}.{patch}"


# ----------------------------------------------------------
# 3. Version history class
# ----------------------------------------------------------

class ModelVersionHistory:

    def __init__(self):
        self.history = []

    def record(self, version, change_type, description, metrics):

        entry = {
            "version": version,
            "change_type": change_type,
            "description": description,
            "metrics": metrics,
            "timestamp": datetime.now().isoformat()
        }

        self.history.append(entry)

    def get_changelog(self):

        lines = []

        for h in self.history:
            line = f"{h['version']} ({h['change_type']}) - {h['description']} | F1={h['metrics'].get('f1',0):.4f}"
            lines.append(line)

        return "\n".join(lines)

    def get_version_for_rollback(self, n_versions_back):

        if len(self.history) <= n_versions_back:
            raise ValueError("Not enough versions for rollback")

        return self.history[-(n_versions_back + 1)]["version"]


# ----------------------------------------------------------
# 4. Save current best model
# ----------------------------------------------------------

client = MlflowClient()

best_run = client.get_run(BEST_RUN_ID)

best_model = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")

best_metrics = best_run.data.metrics
best_params  = best_run.data.params

model_path, meta = save_versioned_model(
    best_model,
    "loan-default-detector",
    "v1.0.0",
    best_metrics,
    best_params,
    dataset_version="v1",
    run_id=BEST_RUN_ID
)

print("Model saved at:", model_path)


# ----------------------------------------------------------
# 5. Track version history
# ----------------------------------------------------------

history = ModelVersionHistory()

history.record(
    version="v1.0.0",
    change_type="major",
    description="Initial release from MLflow best run",
    metrics=best_metrics
)

print("\nVersion changelog:")
print(history.get_changelog())


# ----------------------------------------------------------
# 6. Register model in MLflow Registry
# ----------------------------------------------------------

REGISTRY_NAME = "LoanDefaultDetector"

model_uri = f"runs:/{BEST_RUN_ID}/model"

registered = mlflow.register_model(model_uri, REGISTRY_NAME)

client.update_registered_model(
    name=REGISTRY_NAME,
    description="Loan default prediction model used by LoanGuard"
)

client.set_registered_model_tag(
    name=REGISTRY_NAME,
    key="semantic_version",
    value="v1.0.0"
)

print("\nRegistered model version:", registered.version)


# ----------------------------------------------------------
# 7. Simulate a new version (patch update)
# ----------------------------------------------------------

new_version = bump_version("v1.0.0", "patch")

history.record(
    version=new_version,
    change_type="patch",
    description="Threshold adjustment for improved recall",
    metrics={"f1": best_metrics["f1"] - 0.02}  # simulate worse performance
)

print("\nUpdated changelog:")
print(history.get_changelog())


# ----------------------------------------------------------
# 8. Demonstrate rollback
# ----------------------------------------------------------

rollback_version = history.get_version_for_rollback(1)

print("\nProduction failure detected!")
print("Rolling back to:", rollback_version)


rollback_path = LAB_DIR / "models" / f"loan-default-detector-{rollback_version}" / "model.pkl"

with open(rollback_path, "rb") as f:
    rollback_model = pickle.load(f)

rollback_preds = rollback_model.predict(X_test)

rollback_f1 = f1_score(y_test, rollback_preds)

print("\nRollback model F1:", rollback_f1)
print("Rollback successful — performance restored.")

Registered model 'LoanDefaultDetector' already exists. Creating a new version of this model...
2026/03/13 13:46:25 WARNING mlflow.tracking._model_registry.fluent: Run with id fcdf035dcfab4f2da92566324fb6e02d has no artifacts at artifact path 'model', registering model based on models:/m-8d5905685169456e80edecf3c06f10ab instead


Model saved at: loanGuard_lab/models/loan-default-detector-v1.0.0

Version changelog:
v1.0.0 (major) - Initial release from MLflow best run | F1=0.8783

Registered model version: 7

Updated changelog:
v1.0.0 (major) - Initial release from MLflow best run | F1=0.8783
v1.0.1 (patch) - Threshold adjustment for improved recall | F1=0.8583

Production failure detected!
Rolling back to: v1.0.0

Rollback model F1: 0.8782608695652174
Rollback successful — performance restored.


Created version '7' of model 'LoanDefaultDetector'.


---
# TASK 4 - Registry Lifecycle Management
> Staging → Production → Archived: govern your models
---

##  Task 4 - Registry Lifecycle - Staging, Production, Archived

### Scenario
> LoanGuard just hired a new senior ML engineer. She asks: 'What model is in production right now? When was it approved? Who approved it? What does rollback look like?' Nobody can answer. In this task you'll implement the full governance lifecycle - every model moves through defined stages with approvals and audit logs.

### Objective
By the end of this task you will be able to:
- Transition a model through Staging → Production → Archived stages in MLflow
- Write an audit log that records every lifecycle event
- Implement a rollback function that demotes the current production model
- Query the registry to answer governance questions programmatically

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  4A - Transition model through lifecycle stages

Move the registered model through the stages: None → Staging → Production

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `MlflowClient().transition_model_version_stage(name, version, stage)` where stage is 'Staging', 'Production', or 'Archived'.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

After each transition, add a comment: `client.update_model_version(name, version, description=...)` to record the reason.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
client = MlflowClient()
client.transition_model_version_stage(
    name    = REGISTRY_NAME,
    version = "1",
    stage   = "Staging",
    archive_existing_versions=False
)
```

</details>

In [79]:
#  BEGINNER 4A - Lifecycle stage transitions
# ---------------------------------------------------------

client = MlflowClient()

def transition_stage(name, version, stage, reason=""):
    """
    Transition a registered model version to a new stage and record the reason.

    Parameters
    ----------
    name    : str  registered model name
    version : str  model version number as string e.g. '1'
    stage   : str  'Staging', 'Production', or 'Archived'
    reason  : str  human-readable reason for the transition
    """

    # Step 1: Transition the model stage
    client.transition_model_version_stage(
        name=name,
        version=version,
        stage=stage,
        archive_existing_versions=False
    )

    # Step 2: Update the model version description with the reason
    client.update_model_version(
        name=name,
        version=version,
        description=reason
    )

    print(f"  {name} v{version}  →  {stage}")
    if reason:
        print(f"Reason: {reason}")


# ----- Walk the model through its lifecycle -----
latest = client.get_latest_versions(REGISTRY_NAME)

if latest:
    ver = latest[0].version

    print("Stage transitions for LoanDefaultDetector:")
    print("─" * 50)

    transition_stage(
        REGISTRY_NAME, ver, "Staging",
        reason="Passed unit tests and offline evaluation - F1 > 0.72 threshold met"
    )

    # transition to Production
    transition_stage(
        REGISTRY_NAME, ver, "Production",
        reason="Approved by ML review board after successful staging validation and monitoring checks"
    )

    # Verify current stage
    current = client.get_model_version(REGISTRY_NAME, ver)

    print(f"\nCurrent stage: {current.current_stage}")
    print(f"Description  : {current.description}")

Stage transitions for LoanDefaultDetector:
──────────────────────────────────────────────────
  LoanDefaultDetector v7  →  Staging
Reason: Passed unit tests and offline evaluation - F1 > 0.72 threshold met
  LoanDefaultDetector v7  →  Production
Reason: Approved by ML review board after successful staging validation and monitoring checks

Current stage: Production
Description  : Approved by ML review board after successful staging validation and monitoring checks


###  4B - Build an audit log

Every lifecycle event should be recorded. Implement an audit log that tracks: timestamp, model name, version, old stage, new stage, and the actor/reason.

In [80]:
    #  BEGINNER 4B - Audit log
# ---------------------------------------------------------

AUDIT_LOG = []   # list of event dicts

def log_audit_event(model_name, version, from_stage, to_stage,
                     actor, reason):
    """
    Record a lifecycle event in the audit log and persist to disk.
    """

    event = {
        "timestamp":    datetime.now().isoformat(),
        "model_name":   model_name,
        "version":      version,
        "from_stage":   from_stage,
        "to_stage":     to_stage,
        "actor":        actor,
        "reason":       reason
    }

    # Step 1: Append to in-memory log
    AUDIT_LOG.append(event)

    # Step 2: Save to JSON file (append-style - read existing, add, write back)
    log_path = LAB_DIR / "audit_log.json"

    if log_path.exists():
        with open(log_path, "r") as f:
            existing = json.load(f)
    else:
        existing = []

    existing.append(event)

    with open(log_path, "w") as f:
        json.dump(existing, f, indent=2)

    return event


def print_audit_log():
    """Print a formatted audit log table."""
    if not AUDIT_LOG:
        print("Audit log is empty")
        return

    print(f"{'Timestamp':<26} {'Model':<22} {'v':<4} {'From':<12} {'To':<12} {'Actor':<18} Reason")
    print("─" * 110)

    for e in AUDIT_LOG:
        ts = e['timestamp'][:19]

        print(f"{ts:<26} {e['model_name']:<22} {e['version']:<4} "
              f"{e['from_stage']:<12} {e['to_stage']:<12} {e['actor']:<18} {e['reason']}")


# ----- Replay lifecycle events through audit log -----

ver = client.get_latest_versions(REGISTRY_NAME)[0].version

log_audit_event(
    REGISTRY_NAME,
    ver,
    "None",
    "Staging",
    "ci-system",
    "Automated: offline eval passed F1 threshold"
)

log_audit_event(
    REGISTRY_NAME,
    ver,
    "Staging",
    "Production",
    "ml-lead@loanGuard.ai",
    "Manual approval: A/B test shows +3% approval lift"
)

print("LoanGuard Model Audit Log:")
print("=" * 110)

print_audit_log()

LoanGuard Model Audit Log:
Timestamp                  Model                  v    From         To           Actor              Reason
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
2026-03-13T13:46:34        LoanDefaultDetector    6    None         Staging      ci-system          Automated: offline eval passed F1 threshold
2026-03-13T13:46:34        LoanDefaultDetector    6    Staging      Production   ml-lead@loanGuard.ai Manual approval: A/B test shows +3% approval lift


###  4C - Rollback

Simulate a production failure: a new model version is promoted, starts failing, and you need to roll back to the previous production model.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

To rollback: archive the current production version, then transition the previous version back to Production.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Use `client.search_model_versions(f"name='{name}'")` to find all versions. Filter by `current_stage == 'Archived'` to find previous production candidates.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def rollback_production(name, rollback_to_version, reason):
    # 1. Find and archive current production version
    current_prod = [v for v in client.search_model_versions(f"name='{name}'")
                    if v.current_stage == 'Production']
    if current_prod:
        client.transition_model_version_stage(name, current_prod[0].version, 'Archived')
    # 2. Promote rollback target to Production
    client.transition_model_version_stage(name, rollback_to_version, 'Production')
```

</details>

In [81]:
#  BEGINNER 4C - Simulate promotion of v2, then rollback
# ---------------------------------------------------------

# First: Register a second (worse) model version to simulate a bad deployment
print("Simulating a bad deployment...")

# Train a deliberately weak model
weak_model = LogisticRegression(C=0.001, max_iter=100, random_state=42)

with mlflow.start_run(run_name="WeakModel_bad_deploy") as bad_run:
    weak_model.fit(X_train, y_train)

    weak_preds = weak_model.predict(X_test)

    mlflow.log_metric("f1", f1_score(y_test, weak_preds))
    mlflow.set_tag("dataset_version", "v1")

    mlflow.sklearn.log_model(weak_model, "model")

    bad_run_id = bad_run.info.run_id


# Register the weak model as version 2
bad_registered = mlflow.register_model(
    f"runs:/{bad_run_id}/model",
    REGISTRY_NAME
)

bad_version = bad_registered.version


# Promote bad model to Production (simulating mistaken deployment)
client.transition_model_version_stage(
    REGISTRY_NAME,
    bad_version,
    "Production",
    archive_existing_versions=True
)

log_audit_event(
    REGISTRY_NAME,
    bad_version,
    "Staging",
    "Production",
    "auto-deploy-pipeline",
    "Automated promotion - metric check bypassed (BUG)"
)

print(f"Bad model (version {bad_version}) promoted to Production - F1: {f1_score(y_test, weak_preds):.4f}")
print(" Production is now degraded!")

print()
print("─" * 50)
print("Initiating rollback...")
print("─" * 50)


def rollback_production(registry_name, rollback_to_version, reason, actor):
    """
    Roll back production to a previously archived version.

    Steps:
      1. Find the current Production version and move it to Archived
      2. Move rollback_to_version back to Production
      3. Log both events in the audit log
    """

    # Step 1: Find current production version
    current_prod_versions = [
        v for v in client.search_model_versions(f"name='{registry_name}'")
        if v.current_stage == "Production"
    ]

    # Step 2: Archive current production
    for v in current_prod_versions:

        client.transition_model_version_stage(
            name=registry_name,
            version=v.version,
            stage="Archived"
        )

        log_audit_event(
            registry_name,
            v.version,
            "Production",
            "Archived",
            actor,
            f"ROLLBACK: replaced by v{rollback_to_version}. {reason}"
        )

    # Step 3: Promote rollback target back to Production
    client.transition_model_version_stage(
        name=registry_name,
        version=rollback_to_version,
        stage="Production"
    )

    log_audit_event(
        registry_name,
        rollback_to_version,
        "Archived",
        "Production",
        actor,
        f"ROLLBACK TARGET: {reason}"
    )

    print(f" Rolled back to version {rollback_to_version}")


# Execute rollback to version 1
v1_version = client.get_latest_versions(REGISTRY_NAME, stages=["Archived"])

if v1_version:
    rollback_production(
        registry_name       = REGISTRY_NAME,
        rollback_to_version = v1_version[0].version,
        reason              = "Production F1 dropped from 0.72 to 0.41 after v2 deployment",
        actor               = "ml-lead@loanGuard.ai"
    )

print()
print("Final Audit Log:")
print("=" * 110)

print_audit_log()

Simulating a bad deployment...


2026/03/13 13:46:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 13:46:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'LoanDefaultDetector' already exists. Creating a new version of this model...
2026/03/13 13:46:52 WARNING mlflow.tracking._model_registry.fluent: Run with id 4524bde0b79a4ad8b163d19a3ccd5894 has no artifacts at artifact path 'model', registering model based on models:/m-13d4aac15c8e45d384242eeb3c5d49d6 instead


Bad model (version 8) promoted to Production - F1: 0.6936
 Production is now degraded!

──────────────────────────────────────────────────
Initiating rollback...
──────────────────────────────────────────────────
 Rolled back to version 7

Final Audit Log:
Timestamp                  Model                  v    From         To           Actor              Reason
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
2026-03-13T13:46:34        LoanDefaultDetector    6    None         Staging      ci-system          Automated: offline eval passed F1 threshold
2026-03-13T13:46:34        LoanDefaultDetector    6    Staging      Production   ml-lead@loanGuard.ai Manual approval: A/B test shows +3% approval lift
2026-03-13T13:46:52        LoanDefaultDetector    8    Staging      Production   auto-deploy-pipeline Automated promotion - metric check bypassed (BUG)
2026-03-13T13:46:52        LoanDefaultDetector    8    Production   Archived 

Created version '8' of model 'LoanDefaultDetector'.


###  4D - Governance Query

Answer the governance questions that LoanGuard's new engineer asked - programmatically.

In [54]:
#  BEGINNER 4D - Answer governance questions programmatically
# ------------------------------------------------------------

print("╔══════════════════════════════════════════════════════╗")
print("║        LoanGuard Model Governance Report             ║")
print("╚══════════════════════════════════════════════════════╝")
print()

# Q1: What model is in production right now?
prod_models = [
    v for v in client.search_model_versions(f"name='{REGISTRY_NAME}'")
    if v.current_stage == "Production"
]

print(f"Q1 - Current production model:")

for m in prod_models:
    print(f"     {m.name}  version {m.version}  ({m.current_stage})")

print()


# Q2: When was it promoted to production?
prod_event = None

for e in reversed(AUDIT_LOG):
    if e["to_stage"] == "Production":
        prod_event = e
        break

print(f"Q2 - Promoted to production at: {prod_event['timestamp'] if prod_event else 'Unknown'}")

print()


# Q3: Who approved it?
actor = prod_event["actor"] if prod_event else "Unknown"

print(f"Q3 - Approved by: {actor}")

print()


# Q4: What data trained it?
dataset_version = "Unknown"

if prod_models:
    run_id = client.get_model_version(REGISTRY_NAME, prod_models[0].version).run_id

    run_data = client.get_run(run_id)

    dataset_version = run_data.data.tags.get("dataset_version", "Unknown")

print(f"Q4 - Training dataset: {dataset_version}")

print()


# Q5: What were the production model's metrics?
metrics = {}

if prod_models:
    run_id = client.get_model_version(REGISTRY_NAME, prod_models[0].version).run_id
    run_data = client.get_run(run_id)
    metrics = run_data.data.metrics

print(f"Q5 - Metrics at registration:")
for k, v in metrics.items():
    print(f"     {k}: {v:.4f}")

╔══════════════════════════════════════════════════════╗
║        LoanGuard Model Governance Report             ║
╚══════════════════════════════════════════════════════╝

Q1 - Current production model:
     LoanDefaultDetector  version 2  (Production)

Q2 - Promoted to production at: 2026-03-13T09:48:36.869378

Q3 - Approved by: ml-lead@loanGuard.ai

Q4 - Training dataset: v1

Q5 - Metrics at registration:
     accuracy: 0.8600
     precision: 0.8860
     recall: 0.8707
     f1: 0.8783
     auc: 0.9369


In [82]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert len(AUDIT_LOG) >= 2, 'Audit log must have at least 2 events'
    passed.append('Audit log has entries')
except Exception as e:
    failed.append(('Audit log has entries', str(e)))

try:
    assert all('timestamp' in e for e in AUDIT_LOG), 'Each event needs a timestamp'
    passed.append('Audit log has timestamps')
except Exception as e:
    failed.append(('Audit log has timestamps', str(e)))

try:
    assert (LAB_DIR / 'audit_log.json').exists(), 'audit_log.json not written to disk'
    passed.append('Audit log persisted')
except Exception as e:
    failed.append(('Audit log persisted', str(e)))

try:
    assert len(client.get_latest_versions(REGISTRY_NAME, stages=['Production'])) > 0, 'No model in Production stage'
    passed.append('Production model exists')
except Exception as e:
    failed.append(('Production model exists', str(e)))

try:
    assert len([e for e in AUDIT_LOG if 'ROLLBACK' in e.get('reason','')]) > 0, 'No rollback event in audit log'
    passed.append('Rollback executed')
except Exception as e:
    failed.append(('Rollback executed', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 5/5
  Audit log has entries
  Audit log has timestamps
  Audit log persisted
  Production model exists
  Rollback executed

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 4

Build a complete governance system:

**Requirements:**
1. `transition_stage()` with stage validation (only allow valid stage progressions: None→Staging, Staging→Production, Production→Archived)
2. `log_audit_event()` that writes to both in-memory list and a JSON file on disk
3. `rollback_production()` - archives current prod, promotes target, logs both events
4. `GovernanceReport` class with methods:
   - `current_production()` - returns name, version, stage, promoter, promotion time
   - `full_changelog(model_name)` - returns complete event history from audit log
   - `models_in_stage(stage)` - returns all models currently in a given stage
5. Simulate the full LoanGuard lifecycle: register v1 → staging → production → register bad v2 → production → rollback → archive v2
6. Print the final governance report answering all 5 questions from 4D

**Bonus:** Add a gate that blocks promotion to Production if the model's F1 score (from MLflow metrics) is below 0.65

In [83]:
# ADVANCED - Task 4 Full Implementation
# ----------------------------------------------------------

import json
from datetime import datetime
from pathlib import Path
from mlflow.tracking import MlflowClient

client = MlflowClient()

AUDIT_LOG = []
AUDIT_PATH = LAB_DIR / "audit_log.json"

VALID_TRANSITIONS = {
    "None": ["Staging"],
    "Staging": ["Production"],
    "Production": ["Archived"]
}


# ----------------------------------------------------------
# Audit logging
# ----------------------------------------------------------

def log_audit_event(model_name, version, from_stage, to_stage, actor, reason):

    event = {
        "timestamp": datetime.now().isoformat(),
        "model_name": model_name,
        "version": version,
        "from_stage": from_stage,
        "to_stage": to_stage,
        "actor": actor,
        "reason": reason
    }

    AUDIT_LOG.append(event)

    if AUDIT_PATH.exists():
        with open(AUDIT_PATH, "r") as f:
            existing = json.load(f)
    else:
        existing = []

    existing.append(event)

    with open(AUDIT_PATH, "w") as f:
        json.dump(existing, f, indent=2)

    return event


# ----------------------------------------------------------
# Stage transition with validation
# ----------------------------------------------------------

def transition_stage(name, version, stage, actor, reason):

    mv = client.get_model_version(name, version)

    current_stage = mv.current_stage or "None"

    if stage not in VALID_TRANSITIONS.get(current_stage, []):
        raise ValueError(f"Invalid transition {current_stage} → {stage}")

    # BONUS: production gate
    if stage == "Production":

        run = client.get_run(mv.run_id)
        f1 = run.data.metrics.get("f1", 0)

        if f1 < 0.65:
            raise RuntimeError(f"Promotion blocked: F1={f1:.3f} below 0.65 threshold")

    client.transition_model_version_stage(
        name=name,
        version=version,
        stage=stage,
        archive_existing_versions=False
    )

    log_audit_event(name, version, current_stage, stage, actor, reason)


# ----------------------------------------------------------
# Rollback logic
# ----------------------------------------------------------

def rollback_production(name, target_version, actor, reason):

    prod_versions = [
        v for v in client.search_model_versions(f"name='{name}'")
        if v.current_stage == "Production"
    ]

    for v in prod_versions:

        client.transition_model_version_stage(
            name=name,
            version=v.version,
            stage="Archived"
        )

        log_audit_event(
            name,
            v.version,
            "Production",
            "Archived",
            actor,
            f"ROLLBACK: replaced by v{target_version}. {reason}"
        )

    client.transition_model_version_stage(
        name=name,
        version=target_version,
        stage="Production"
    )

    log_audit_event(
        name,
        target_version,
        "Archived",
        "Production",
        actor,
        f"ROLLBACK TARGET: {reason}"
    )


# ----------------------------------------------------------
# Governance reporting
# ----------------------------------------------------------

class GovernanceReport:

    def current_production(self):

        prod = [
            v for v in client.search_model_versions("")
            if v.current_stage == "Production"
        ]

        if not prod:
            return None

        mv = prod[0]

        event = next(
            (e for e in reversed(AUDIT_LOG)
             if e["version"] == mv.version and e["to_stage"] == "Production"),
            None
        )

        return {
            "name": mv.name,
            "version": mv.version,
            "stage": mv.current_stage,
            "promoter": event["actor"] if event else None,
            "promotion_time": event["timestamp"] if event else None
        }

    def full_changelog(self, model_name):

        events = [e for e in AUDIT_LOG if e["model_name"] == model_name]

        lines = []

        for e in events:
            lines.append(
                f"{e['timestamp']} | v{e['version']} | {e['from_stage']}→{e['to_stage']} | {e['actor']} | {e['reason']}"
            )

        return "\n".join(lines)

    def models_in_stage(self, stage):

        return [
            (v.name, v.version)
            for v in client.search_model_versions("")
            if v.current_stage == stage
        ]


# ----------------------------------------------------------
# Lifecycle simulation
# ----------------------------------------------------------

print("Simulating LoanGuard lifecycle")

# assume BEST_RUN_ID already exists

model_uri = f"runs:/{BEST_RUN_ID}/model"

reg = mlflow.register_model(model_uri, REGISTRY_NAME)
v1 = reg.version

transition_stage(
    REGISTRY_NAME,
    v1,
    "Staging",
    "ci-system",
    "Offline evaluation passed"
)

transition_stage(
    REGISTRY_NAME,
    v1,
    "Production",
    "ml-lead@loanGuard.ai",
    "Manual approval after review"
)


# simulate bad model v2

weak_model = LogisticRegression(C=0.001, max_iter=100)

with mlflow.start_run() as r:

    weak_model.fit(X_train, y_train)

    preds = weak_model.predict(X_test)

    mlflow.log_metric("f1", f1_score(y_test, preds))

    mlflow.sklearn.log_model(weak_model, "model")

    bad_run = r.info.run_id

bad_reg = mlflow.register_model(f"runs:/{bad_run}/model", REGISTRY_NAME)
v2 = bad_reg.version

transition_stage(
    REGISTRY_NAME,
    v2,
    "Staging",
    "ci-system",
    "Auto deploy"
)

# simulate mistaken production deploy

client.transition_model_version_stage(
    REGISTRY_NAME,
    v2,
    "Production",
    archive_existing_versions=True
)

log_audit_event(
    REGISTRY_NAME,
    v2,
    "Staging",
    "Production",
    "auto-deploy",
    "Pipeline bug"
)


# rollback

rollback_production(
    REGISTRY_NAME,
    v1,
    "ml-lead@loanGuard.ai",
    "Production metrics dropped"
)


# archive bad model

client.transition_model_version_stage(
    REGISTRY_NAME,
    v2,
    "Archived"
)

log_audit_event(
    REGISTRY_NAME,
    v2,
    "Production",
    "Archived",
    "ml-lead@loanGuard.ai",
    "Retired after rollback"
)


# ----------------------------------------------------------
# Final governance report
# ----------------------------------------------------------

report = GovernanceReport()

print("\n GOVERNANCE REPORT\n")

prod = report.current_production()

print("Q1 - Current production model:")
print(prod)

print("\nQ2/Q3 - Promotion details:")
print(f"Promoted by {prod['promoter']} at {prod['promotion_time']}")

print("\nQ4 - Models in Production:")
print(report.models_in_stage("Production"))

print("\nQ5 - Full changelog:")
print(report.full_changelog(REGISTRY_NAME))

Registered model 'LoanDefaultDetector' already exists. Creating a new version of this model...
2026/03/13 13:46:59 WARNING mlflow.tracking._model_registry.fluent: Run with id fcdf035dcfab4f2da92566324fb6e02d has no artifacts at artifact path 'model', registering model based on models:/m-8d5905685169456e80edecf3c06f10ab instead


Simulating LoanGuard lifecycle


Created version '9' of model 'LoanDefaultDetector'.
2026/03/13 13:46:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 13:46:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'LoanDefaultDetector' already exists. Creating a new version of this model...
2026/03/13 13:47:01 WARNING mlflow.tracking._model_registry.fluent: Run with id 16023d064f2e4392ba07e9ff4b64da27 has no artifacts at artifact path 'model', registering model based on models:/m-ff5d7f43e4c447a8a9bf6b521cfcd5a1 instead



 GOVERNANCE REPORT

Q1 - Current production model:
{'name': 'LoanDefaultDetector', 'version': 9, 'stage': 'Production', 'promoter': 'ml-lead@loanGuard.ai', 'promotion_time': '2026-03-13T13:47:01.995834'}

Q2/Q3 - Promotion details:
Promoted by ml-lead@loanGuard.ai at 2026-03-13T13:47:01.995834

Q4 - Models in Production:
[('LoanDefaultDetector', 9)]

Q5 - Full changelog:
2026-03-13T13:46:59.095063 | v9 | None→Staging | ci-system | Offline evaluation passed
2026-03-13T13:46:59.118797 | v9 | Staging→Production | ml-lead@loanGuard.ai | Manual approval after review
2026-03-13T13:47:01.949580 | v10 | None→Staging | ci-system | Auto deploy
2026-03-13T13:47:01.964132 | v10 | Staging→Production | auto-deploy | Pipeline bug
2026-03-13T13:47:01.982567 | v10 | Production→Archived | ml-lead@loanGuard.ai | ROLLBACK: replaced by v9. Production metrics dropped
2026-03-13T13:47:01.995834 | v9 | Archived→Production | ml-lead@loanGuard.ai | ROLLBACK TARGET: Production metrics dropped
2026-03-13T13:47:0

Created version '10' of model 'LoanDefaultDetector'.


---
#  Lab Complete - Reflection
---

##  What You Built

| Task | What you implemented | Real-world equivalent |
|------|----------------------|----------------------|
| **Task 1** | DVC pointer files, drift detection, schema validator | DVC + Great Expectations |
| **Task 2** | MLflow experiment runs, multi-model comparison, artifact retrieval | MLflow Tracking Server |
| **Task 3** | Versioned model artifacts, semantic version logic, model registration | MLflow Registry + release process |
| **Task 4** | Stage transitions, audit log, rollback, governance queries | MLflow Registry Lifecycle + compliance |

---

## The Full Stack - How It Connects

```
loans_v1.csv  →  validate_schema()  →  [PASSES]  →  Training Run
                                                         │
                                              mlflow.log_params()
                                              mlflow.log_metrics()
                                              mlflow.log_model()
                                                         │
                                              register_model("LoanDefaultDetector")
                                                         │
                                         None → Staging → Production
                                                    (with audit log)
                                                         │
                             If it fails →  rollback_production()  →  Archived
```

---

##  Reflection Questions

Answer these in the cell below before the session ends:

1. Which task felt most like something you'd use immediately in your current project?
2. Where in LoanGuard's original problem could a schema validator have saved the most time?
3. If you had to pick ONE layer of the versioning stack to implement tomorrow - which would it be and why?

In [84]:
#  Your Reflection Answers
# ----------------------------------------------------

reflection = {
    "most_immediately_useful": """
    The experiment tracking with MLflow felt most useful to me. 
    It helps keep track of different model runs, parameters, and results in one place. 
    This makes it easier to compare models and choose the best one instead of guessing.
    """,

    "where_schema_validation_would_have_helped": """
    A schema validator would have helped when the new dataset version was introduced. 
    It could quickly detect issues like missing columns, extra columns, or too many null values 
    before the model training started.
    """,

    "first_layer_to_implement": """
    I would choose experiment tracking first. 
    It records all experiments, parameters, and metrics in one place, 
    so the team can easily understand which model performed best and reproduce the results later.
    """
}

for q, a in reflection.items():
    print(f"Q: {q}")
    print(f"A: {a.strip()}")
    print()

Q: most_immediately_useful
A: The experiment tracking with MLflow felt most useful to me. 
    It helps keep track of different model runs, parameters, and results in one place. 
    This makes it easier to compare models and choose the best one instead of guessing.

Q: where_schema_validation_would_have_helped
A: A schema validator would have helped when the new dataset version was introduced. 
    It could quickly detect issues like missing columns, extra columns, or too many null values 
    before the model training started.

Q: first_layer_to_implement
A: I would choose experiment tracking first. 
    It records all experiments, parameters, and metrics in one place, 
    so the team can easily understand which model performed best and reproduce the results later.

